# Bronze Layer — Raw Ingestion

Ingests source files as-is into Delta tables with no transformation.
Schema enforcement is intentionally minimal here — the goal is to preserve
the raw data exactly as received, so Silver always has a full audit trail to
recover from if cleaning logic changes later.

In [ ]:
# Databricks: cluster must have Delta Lake runtime (included by default)
# Local: pip install pyspark delta-spark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from datetime import datetime, timezone

spark = SparkSession.builder.appName("amsterdam-rental-bronze").getOrCreate()

# Adjust these paths when running on Databricks (use /Volumes or DBFS paths)
AIRBNB_SRC   = "../data/airbnb.csv"
RENTALS_SRC  = "../data/rentals.json"
BRONZE_PATH  = "../data/output/bronze"

INGESTED_AT = datetime.now(timezone.utc).isoformat()

## Airbnb — bronze

In [ ]:
airbnb_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")   # keep everything as strings at bronze
    .csv(AIRBNB_SRC)
    .withColumn("_ingested_at", F.lit(INGESTED_AT))
    .withColumn("_source_file", F.lit("airbnb.csv"))
)

print(f"Airbnb rows: {airbnb_raw.count()}")
airbnb_raw.printSchema()
airbnb_raw.show(3, truncate=False)

In [ ]:
(
    airbnb_raw
    .write
    .format("delta")
    .mode("overwrite")
    .save(f"{BRONZE_PATH}/airbnb")
)
print("Airbnb bronze written.")

## Rentals — bronze

In [ ]:
rentals_raw = (
    spark.read
    .option("multiline", "true")
    .json(RENTALS_SRC)
    .withColumn("_ingested_at", F.lit(INGESTED_AT))
    .withColumn("_source_file", F.lit("rentals.json"))
)

print(f"Rentals rows: {rentals_raw.count()}")
rentals_raw.printSchema()
rentals_raw.show(3, truncate=True)

In [ ]:
(
    rentals_raw
    .write
    .format("delta")
    .mode("overwrite")
    .save(f"{BRONZE_PATH}/rentals")
)
print("Rentals bronze written.")

## Validation checks

In [ ]:
airbnb_bronze  = spark.read.format("delta").load(f"{BRONZE_PATH}/airbnb")
rentals_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/rentals")

assert airbnb_bronze.count()  == 9913,  f"Unexpected Airbnb row count"
assert rentals_bronze.count() == 46722, f"Unexpected Rentals row count"

print("Bronze validation passed.")